# Demo: Solar Panel Detection with GeoAI

This **demo** uses the [GeoAI](https://opengeoai.org/) package and a **pre-trained** solar panel detector to find solar panels in NAIP imagery (Davis, CA). No model training—run inference only. Suitable for Colab free tier.

**Steps:** Install package → Mount Drive & setup → Download sample raster → Visualize → Run detection → Vectorize masks → Filter & visualize results.

---

## 1. Install package & setup

In [ ]:
%pip install geoai-py

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/ABT182_GeoAI"
os.makedirs(f"{BASE_DIR}/data/solar", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)

import geoai
print("BASE_DIR =", BASE_DIR)

## 2. Download sample data

In [ ]:
import shutil

solar_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/solar_panels_davis_ca.tif"
downloaded_file = geoai.download_file(solar_raster_url)
solar_raster_path = f"{BASE_DIR}/data/solar/solar_panels_davis_ca.tif"
shutil.copy(downloaded_file, solar_raster_path)
print("Saved:", solar_raster_path)

## 3. Visualize imagery

In [ ]:
geoai.get_raster_info(solar_raster_path)

In [ ]:
import leafmap
m = leafmap.Map()
m.add_raster(solar_raster_path, layer_name="NAIP (Davis, CA)", bands=[1, 2, 3])
m

## 4. Run solar panel detection

In [ ]:
solar_masks_path = f"{BASE_DIR}/outputs/solar_panel_masks.tif"
detector = geoai.SolarPanelDetector()
solar_masks_path = detector.generate_masks(
    solar_raster_path,
    output_path=solar_masks_path,
    confidence_threshold=0.4,
    mask_threshold=0.5,
    min_object_area=100,
    overlap=0.25,
    chip_size=(400, 400),
    batch_size=4,
    verbose=False,
)
print("Masks saved:", solar_masks_path)

In [ ]:
# View NAIP with predicted solar panel masks (mask is single-band; use indexes=[1])
import leafmap
m = leafmap.Map()
m.add_raster(solar_raster_path, layer_name="NAIP", bands=[1, 2, 3])
m.add_raster(solar_masks_path, layer_name="Solar panel masks", indexes=[1], colormap="autumn", nodata=0)
m

## 5. Vectorize masks & add geometric properties

In [ ]:
solar_vector_path = f"{BASE_DIR}/outputs/solar_panel_masks.geojson"
gdf_solar = geoai.orthogonalize(solar_masks_path, solar_vector_path, epsilon=0.2)
gdf_solar = geoai.add_geometric_properties(gdf_solar, area_unit="m2", length_unit="m")
gdf_solar.head()

## 6. Filter and visualize results

In [ ]:
gdf_solar_filtered = gdf_solar[(gdf_solar["elongation"] < 10) & (gdf_solar["area_m2"] > 5)]
print("Filtered count:", len(gdf_solar_filtered))

In [ ]:
import leafmap
m = leafmap.Map()
m.add_raster(solar_raster_path, layer_name="NAIP", bands=[1, 2, 3])
m.add_data(gdf_solar_filtered, column="area_m2", scheme="Quantiles", cmap="YlOrRd", legend_title="Area (m²)")
m

In [ ]:
import matplotlib.pyplot as plt
gdf_solar_filtered["area_m2"].hist(bins=25, color="steelblue", edgecolor="white")
plt.xlabel("Area (m²)"); plt.ylabel("Count"); plt.title("Distribution of solar panel areas")
plt.tight_layout(); plt.show()
print("Total area (m²):", gdf_solar_filtered["area_m2"].sum())

In [ ]:
gdf_solar_filtered.to_file(f"{BASE_DIR}/outputs/solar_panels_filtered.geojson", driver="GeoJSON")
print("Saved:", f"{BASE_DIR}/outputs/solar_panels_filtered.geojson")

---
## Credits

This demo uses the **[GeoAI](https://opengeoai.org/)** Python package. We thank **Dr. Qiusheng Wu** for creating GeoAI and for the examples that inspired this tutorial.

For more information: **https://opengeoai.org/**